# SkateFormer fuzja joint+bone w detekcji ciągłej (Etap III), PKU-MMD

Kolejnym krokiem było sprawdzenie, czy fuzja dwóch strumieni, joint i bone, poprawia detekcję ciągłą na PKU-MMD. Motywacją była niska precyzja detekcji klas point at person oraz pat on back, mimo przyzwoitych wyników klasyfikacji w Etapie II. Strumień bone koduje relacje między stawami, czyli kąty i długości kości, a nie ich bezwzględne pozycje, więc jego dodanie mogło stłumić część fałszywych alarmów generowanych przez sam strumień joint.
Test objął jedynie konfigurację W=64, S=16, zgodną z tą przyjętą w pracy, w obu protokołach, XSub i XView. Wynik kanoniczny dla samego strumienia joint (63,06% mAP@0,3 dla XSub) pozostał bez zmian, a fuzja jest traktowana jako odrębny, dodatkowy wynik.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/KAIST-VICLab/SkateFormer.git /content/SkateFormer
print('Repozytorium sklonowane.')

Mounted at /content/drive
Repozytorium sklonowane.


In [ ]:
!pip install -q einops timm tensorpack torchpack loguru msgpack msgpack-numpy tabulate tensorboardX
import shutil, numpy as np, os

DRIVE_DATA    = '/content/drive/MyDrive/PKU_data'
DRIVE_WEIGHTS = '/content/drive/MyDrive/SkateFormer_weights/ntu60_CSub'

os.makedirs('/content/SkateFormer/data/ntu', exist_ok=True)

npz_src = f'{DRIVE_DATA}/PKU_XSub_both.npz'
npz_dst = '/content/SkateFormer/data/ntu/PKU_XSub_both.npz'
if not os.path.exists(npz_dst):
    shutil.copy(npz_src, npz_dst)
    print('Skopiowano: PKU_XSub_both.npz')
npz_src = f'{DRIVE_DATA}/PKU_XView_both.npz'
npz_dst = '/content/SkateFormer/data/ntu/PKU_XView_both.npz'
if not os.path.exists(npz_dst):
    shutil.copy(npz_src, npz_dst)
    print('Skopiowano: PKU_XView_both.npz')
npz_src = f'{DRIVE_DATA}/PKU_XSub.npz'
npz_dst = '/content/SkateFormer/data/ntu/PKU_XSub.npz'
if not os.path.exists(npz_dst):
    shutil.copy(npz_src, npz_dst)
    print('Skopiowano: PKU_XSub.npz')
npz_src = f'{DRIVE_DATA}/PKU_XView.npz'
npz_dst = '/content/SkateFormer/data/ntu/PKU_XView.npz'
if not os.path.exists(npz_dst):
    shutil.copy(npz_src, npz_dst)
    print('Skopiowano: PKU_XView.npz')
d = np.load(npz_dst)
print(f'train: {d["x_train"].shape}  test: {d["x_test"].shape}')

pt_src = f'{DRIVE_WEIGHTS}/SkateFormer_j.pt'
pt_dst = '/content/SkateFormer/SkateFormer_j.pt'
if not os.path.exists(pt_dst):
    shutil.copy(pt_src, pt_dst)
    print(f'Skopiowano: SkateFormer_j.pt  ({os.path.getsize(pt_dst)/1024**2:.1f} MB)')
else:
    print('Wagi już istnieją lokalnie.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.5 MB/s eta 0:00:00
Skopiowano: PKU_XSub_both.npz
Skopiowano: PKU_XView_both.npz
Skopiowano: PKU_XSub.npz
Skopiowano: PKU_XView.npz
train: (14268, 300, 150)  test: (7141, 300, 150)
Skopiowano: SkateFormer_j.pt  (13.9 MB)


In [ ]:
import json, shutil
import numpy as np

DRIVE_ROOT    = '/content/drive/MyDrive/PKU_data'
WEIGHTS_DIR   = '/content/drive/MyDrive/SkateFormer_weights/ntu60_CSub'
LOCAL_WEIGHTS_J = '/content/SkateFormer/SkateFormer_j.pt'
LOCAL_WEIGHTS_B = '/content/SkateFormer/SkateFormer_b.pt'
LOCAL_DATA    = '/content/SkateFormer/data/ntu'
OFFICIAL_CFG  = '/content/SkateFormer/config/test/ntu_cs/SkateFormer_j.yaml'
WORK_DIR_ROOT = '/content/SkateFormer/work_dir/pku_bone'

os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(WORK_DIR_ROOT, exist_ok=True)

shutil.copy(f'{WEIGHTS_DIR}/SkateFormer_j.pt', LOCAL_WEIGHTS_J)
shutil.copy(f'{WEIGHTS_DIR}/SkateFormer_b.pt', LOCAL_WEIGHTS_B)
print(f'Wagi joint: {os.path.getsize(LOCAL_WEIGHTS_J)/1024**2:.1f} MB')
print(f'Wagi bone:  {os.path.getsize(LOCAL_WEIGHTS_B)/1024**2:.1f} MB')

SPLIT_DIRS = {'xsub': f'{DRIVE_ROOT}/sweep', 'xview': f'{DRIVE_ROOT}/sweep_xview'}
SPLITS = ['xsub', 'xview']

GT, SEQ_LEN = {}, {}
for sp in SPLITS:
    d = SPLIT_DIRS[sp]
    if not os.path.exists(f'{d}/ground_truth.json'):
        print(f'{sp.upper()}: BRAK danych w {d}')
        continue
    with open(f'{d}/ground_truth.json') as f:
        GT[sp] = json.load(f)
    with open(f'{d}/seq_lengths.json') as f:
        SEQ_LEN[sp] = json.load(f)
    n = 0
    for sid in GT[sp]:
        for inst in GT[sp][sid]:
            if inst[0] == 56:
                inst[0] = 24; n += 1
    print(f'{sp.upper()}: {len(GT[sp])} sekwencji, remap 56->24: {n}')

SPLITS = [sp for sp in SPLITS if sp in GT]
print('Protokoly:', SPLITS)

Wagi joint: 13.9 MB
Wagi bone:  13.9 MB
XSUB: 132 sekwencji, remap 56->24: 143
XVIEW: 359 sekwencji, remap 56->24: 308
Protokoly: ['xsub', 'xview']


Funkcja run_inference przyjmuje argument stream, który określa, czy ładowany jest strumień joint czy bone. Dla strumienia bone w konfiguracji feedera ustawiany jest parametr data_type na wartość b, co uruchamia przeliczenie współrzędnych na wektory kości przez funkcję joint2bone. Wyniki obu strumieni są zapisywane w osobnych katalogach z cache, odpowiednio scores dla joint i scores_bone dla bone

In [ ]:
import pickle, yaml, gc

STREAM_CFG = {
    'j': {'weights': LOCAL_WEIGHTS_J, 'data_type': 'j', 'cache': 'scores'},
    'b': {'weights': LOCAL_WEIGHTS_B, 'data_type': 'b', 'cache': 'scores_bone'},
}

def run_inference(stream, split, W, S, force=False):
    sc = STREAM_CFG[stream]
    drive_sweep = SPLIT_DIRS[split]
    src_npz  = f'{drive_sweep}/PKU_windows_test_W{W}_S{S}.npz'
    dst_npz  = f'{LOCAL_DATA}/PKU_{stream}_{split}_W{W}_S{S}.npz'
    work_dir = f'{WORK_DIR_ROOT}/{stream}_{split}_W{W}_S{S}'
    cache_dir = f'{drive_sweep}/{sc["cache"]}/W{W}_S{S}'
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)

    data = np.load(src_npz)
    n = data['x_test'].shape[0]
    meta = {'seq_id': data['seq_id'], 'start': data['start'],
            'end': data['end'], 'y_ref': data['y_ref']}

    if not os.listdir(work_dir) and os.listdir(cache_dir):
        for f_ in os.listdir(cache_dir):
            shutil.copy(f'{cache_dir}/{f_}', f'{work_dir}/{f_}')
        print(f'  [{stream}] przywrocono z Drive')

    existing = [p for p in os.listdir(work_dir) if p.endswith('score.pkl')]
    if existing and not force:
        print(f'  [{stream}] cache: pomijam inferencje')
    else:
        x_buf = np.zeros((n, W + 1, 150), dtype=np.float32)
        x_buf[:, :W, :] = data['x_test']
        y_test = data['y_test']
        del data; gc.collect()
        np.savez(dst_npz, x_test=x_buf, y_test=y_test)
        del x_buf; gc.collect()

        with open(OFFICIAL_CFG) as f:
            cfg = yaml.safe_load(f)
        cfg['work_dir']   = work_dir
        cfg['weights']    = sc['weights']
        cfg['phase']      = 'test'
        cfg['save_score'] = True
        cfg['test_feeder_args']['data_path'] = dst_npz
        cfg['test_feeder_args']['split']     = 'test'
        cfg['test_feeder_args']['data_type'] = sc['data_type']   # 'j' lub 'b'
        cfg_path = f'{work_dir}/config.yaml'
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f)

        !cd /content/SkateFormer && python main.py --config "{cfg_path}"

        existing = [p for p in os.listdir(work_dir) if p.endswith('score.pkl')]
        assert existing, f'Brak score.pkl w {work_dir}'
        shutil.copy(f'{work_dir}/{existing[0]}', f'{cache_dir}/{existing[0]}')
        os.remove(dst_npz)

    with open(f'{work_dir}/{existing[0]}', 'rb') as f:
        scores = pickle.load(f)
    assert len(scores) == n, f'{len(scores)} vs {n}'
    logits = np.array([scores[f'test_{i}'] for i in range(n)])
    return logits, meta

print('run_inference gotowe (stream j / b).')

run_inference gotowe (stream j / b).


In [ ]:
from collections import defaultdict

INTERACTION_NTU = [49, 50, 51, 52, 53, 54, 55, 57]
MIN_SEG_LEN = 8
NMS_IOU = 0.3
CONF_TH = 0.0

def build_frame_level(logits, meta, seq_lengths, conf_th=CONF_TH):
    p = np.exp(logits - logits.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    fp = {s: np.zeros((T, 60)) for s, T in seq_lengths.items()}
    fc = {s: np.zeros(T) for s, T in seq_lengths.items()}
    for i in range(len(meta['seq_id'])):
        s = str(meta['seq_id'][i])
        st, en = int(meta['start'][i]), int(meta['end'][i])
        fp[s][st:en] += p[i]; fc[s][st:en] += 1
    pred, avg_all = {}, {}
    for s, T in seq_lengths.items():
        avg = fp[s] / np.maximum(fc[s], 1)[:, None]
        pr = np.argmax(avg, 1); cf = np.max(avg, 1)
        pr[cf < conf_th] = -1
        pr[fc[s] == 0] = -1
        pred[s] = pr; avg_all[s] = avg
    return pred, avg_all, fc

def frames_to_segments(pred, prob_avg):
    segs = []; T = len(pred); t = 0
    while t < T:
        c = pred[t]
        if c < 0:
            t += 1; continue
        st = t
        while t < T and pred[t] == c:
            t += 1
        if t - st >= MIN_SEG_LEN:
            segs.append((int(c), st, t, float(prob_avg[st:t, c].mean())))
    return segs

def iou_1d(a, b):
    s = max(a[0], b[0]); e = min(a[1], b[1]); inter = max(0, e - s)
    u = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / u if u > 0 else 0.0

def nms_segments(segs, iou_th=NMS_IOU):
    segs = sorted(segs, key=lambda x: x[3], reverse=True); keep = []
    while segs:
        b = segs.pop(0); keep.append(b)
        segs = [x for x in segs
                if not (x[0] == b[0] and iou_1d((x[1], x[2]), (b[1], b[2])) > iou_th)]
    return keep

def compute_ap(pred_list, gt_list, iou_th):
    if len(gt_list) == 0:
        return None
    preds = sorted(pred_list, key=lambda x: x[3], reverse=True)
    matched = set(); tp = np.zeros(len(preds)); fp_ = np.zeros(len(preds))
    for i, (seq, st, en, sc) in enumerate(preds):
        bi, bj = 0, -1
        for j, (gs, gst, gen) in enumerate(gt_list):
            if gs != seq or (seq, j) in matched:
                continue
            v = iou_1d((st, en), (gst, gen))
            if v > bi:
                bi, bj = v, j
        if bi >= iou_th:
            tp[i] = 1; matched.add((seq, bj))
        else:
            fp_[i] = 1
    tpc = np.cumsum(tp); fpc = np.cumsum(fp_)
    rec = tpc / len(gt_list); prec = tpc / np.maximum(tpc + fpc, 1e-9)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
    for k in range(len(mpre) - 2, -1, -1):
        mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

def evaluate(logits, meta, seq_lengths, gt_segments, conf_th=CONF_TH,
             iou_list=(0.1, 0.3, 0.5), classes=None):
    pred, avg_all, fc = build_frame_level(logits, meta, seq_lengths, conf_th)
    pred_by_cls = defaultdict(list)
    for s in seq_lengths:
        for (c, st, en, sc) in nms_segments(frames_to_segments(pred[s], avg_all[s])):
            pred_by_cls[c].append((s, st, en, sc))
    gt_by_cls = defaultdict(list)
    for s, segs in gt_segments.items():
        T = seq_lengths[s]
        for (c, st, en) in segs:
            st, en = max(0, st), min(T, en)
            if en > st:
                gt_by_cls[c].append((s, st, en))
    if classes is not None:
        gt_by_cls = {c: v for c, v in gt_by_cls.items() if c in classes}
    results = {}
    for iou in iou_list:
        aps = {}
        for c in gt_by_cls:
            ap = compute_ap(pred_by_cls.get(c, []), gt_by_cls[c], iou)
            if ap is not None:
                aps[c] = ap
        results[iou] = {'mAP': float(np.mean(list(aps.values()))) if aps else 0.0,
                        'per_class': aps}
    return results, pred_by_cls, gt_by_cls

def frame_level_accuracy(logits, meta, seq_lengths, gt_segments,
                         conf_th=CONF_TH, classes=None):
    pred, _, _ = build_frame_level(logits, meta, seq_lengths, conf_th)
    ca = ta = 0
    for s, T in seq_lengths.items():
        g = -np.ones(T, int)
        for (c, st, en) in gt_segments[s]:
            if classes is not None and c not in classes:
                continue
            g[max(0, st):min(T, en)] = c
        mk = g >= 0; p = pred[s]
        ca += (p[mk] == g[mk]).sum(); ta += mk.sum()
    return ca / ta if ta else 0.0

print('Funkcje detekcji gotowe.')

Funkcje detekcji gotowe.


Output poniżej zawiera linie "Accuracy", "Top1", "Top5" z main.py SkateFormera — to natywna metryka repozytorium, nieużywana w pracy. Liczy się tylko sanity check i mAP/frame accuracy niżej.

In [ ]:
NAMES = {49:'punch/slap', 50:'kick', 51:'push', 52:'pat on back',
         53:'point at', 54:'hugging', 55:'giving', 57:'handshake'}

def sanity(logits, meta):
    m = meta['y_ref'] >= 0
    return (meta['y_ref'][m] == logits.argmax(1)[m]).mean() * 100

fusion_results = []
per_class_rows = []
for split in SPLITS:
    gt, seq_len = GT[split], SEQ_LEN[split]
    print(f'\n{"="*64}\n{split.upper()} — joint / bone / fusion (W=64, S=16)\n{"="*64}')

    lj, meta = run_inference('j', split, 64, 16)
    lb, _    = run_inference('b', split, 64, 16)

    print(f'  Sanity joint: {sanity(lj, meta):.1f}%   bone: {sanity(lb, meta):.1f}%')
    if sanity(lb, meta) < 10:

    lf = (lj + lb) / 2

    for tag, lg in [('joint', lj), ('bone', lb), ('j+b', lf)]:
        res, _, _   = evaluate(lg, meta, seq_len, gt)
        res_i, _, _ = evaluate(lg, meta, seq_len, gt, classes=INTERACTION_NTU)
        fa   = frame_level_accuracy(lg, meta, seq_len, gt)
        fa_i = frame_level_accuracy(lg, meta, seq_len, gt, classes=INTERACTION_NTU)
        fusion_results.append({
            'split':split, 'stream':tag,
            'frame_acc':fa, 'mAP@0.1':res[0.1]['mAP'], 'mAP@0.3':res[0.3]['mAP'],
            'mAP@0.5':res[0.5]['mAP'], 'frame_acc_int':fa_i,
            'mAP@0.3_int':res_i[0.3]['mAP']})
        print(f'  {tag:5s}: mAP@0,3={res[0.3]["mAP"]*100:5.2f}%  '
              f'interakcje={res_i[0.3]["mAP"]*100:5.2f}%  '
              f'frame={fa*100:5.2f}%')
        # per-klasa interakcji dla fusion
        if tag == 'j+b':
            for c in INTERACTION_NTU:
                per_class_rows.append({'split':split, 'klasa':NAMES[c],
                    'AP_joint':None, 'AP_bone':None, 'AP_jb':res_i[0.3]['per_class'].get(c)})

    #AP joint i bone per-klasa
    rj,_,_ = evaluate(lj, meta, seq_len, gt, classes=INTERACTION_NTU)
    rb,_,_ = evaluate(lb, meta, seq_len, gt, classes=INTERACTION_NTU)
    for row in per_class_rows:
        if row['split'] == split:
            c = [k for k,v in NAMES.items() if v==row['klasa']][0]
            row['AP_joint'] = rj[0.3]['per_class'].get(c)
            row['AP_bone']  = rb[0.3]['per_class'].get(c)

    del lj, lb, lf, meta; import gc; gc.collect()


XSUB — joint / bone / fusion (W=64, S=16)
  [j] przywrocono z Drive
  [j] cache: pomijam inferencje
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
<function SkateFormer_ at 0x78dcf339dc60>
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[ Sat Jul 18 10:07:26 2026 ] Load weights from /content/SkateFormer/SkateFormer_b.pt.
[ Sat Jul 18 10:07:28 2026 ] Model:   model.SkateFormer.SkateFormer_.
[ Sat Jul 18 10:07:28 2026 ] Weights: /content/SkateFormer/SkateFormer_b.pt.
[ Sat Jul 18 10:07:28 2026 ] Ev

In [ ]:
import pandas as pd

dff = pd.DataFrame(fusion_results)
for c in ['frame_acc','mAP@0.1','mAP@0.3','mAP@0.5','frame_acc_int','mAP@0.3_int']:
    dff[c] = (dff[c]*100).round(2)
print('=== Podsumowanie: joint vs bone vs fusion ===')
display(dff)

dpc = pd.DataFrame(per_class_rows)
for c in ['AP_joint','AP_bone','AP_jb']:
    dpc[c] = (dpc[c].astype(float)*100).round(1)
print('\n=== AP@0,3 per klasa interakcji (fusion vs skladowe) ===')
for sp in SPLITS:
    print(f'\n{sp.upper()}:')
    display(dpc[dpc.split==sp][['klasa','AP_joint','AP_bone','AP_jb']].reset_index(drop=True))

=== Podsumowanie: joint vs bone vs fusion ===


,split,stream,frame_acc,mAP@0.1,mAP@0.3,mAP@0.5,frame_acc_int,mAP@0.3_int
0,xsub,joint,55.57,73.04,63.06,44.32,70.87,57.44
1,xsub,bone,59.82,76.27,67.86,47.27,68.44,64.19
2,xsub,j+b,60.62,77.74,68.51,48.19,72.13,63.29
3,xview,joint,57.69,74.23,64.18,44.09,73.59,61.79
4,xview,bone,61.90,77.45,69.16,48.38,70.02,67.48
5,xview,j+b,62.67,78.80,69.97,48.86,74.72,67.91



=== AP@0,3 per klasa interakcji (fusion vs skladowe) ===

XSUB:


,klasa,AP_joint,AP_bone,AP_jb
0,punch/slap,42.2,47.6,36.8
1,kick,44.3,81.7,65.3
2,push,67.6,68.2,80.5
3,pat on back,8.8,2.6,8.7
4,point at,6.1,26.2,23.5
5,hugging,95.4,99.7,100.0
6,giving,99.5,91.7,95.7
7,handshake,95.7,95.8,95.8



XVIEW:


,klasa,AP_joint,AP_bone,AP_jb
0,punch/slap,66.6,75.9,73.0
1,kick,58.8,82.4,78.2
2,push,61.9,60.3,63.5
3,pat on back,10.3,1.8,5.9
4,point at,11.3,28.3,30.4
5,hugging,92.9,99.9,99.9
6,giving,95.3,94.1,93.9
7,handshake,97.1,97.1,98.6


## KROK 7 — Zapis

In [ ]:
dff.to_csv(f'{DRIVE_ROOT}/bone_fusion_results.csv', index=False)
dpc.to_csv(f'{DRIVE_ROOT}/bone_fusion_per_class.csv', index=False)
print('Zapisano:')
print(f'  {DRIVE_ROOT}/bone_fusion_results.csv')
print(f'  {DRIVE_ROOT}/bone_fusion_per_class.csv')
print(f'  scores_bone/ na Drive (cache bone)')

Zapisano:
  /content/drive/MyDrive/PKU_data/bone_fusion_results.csv
  /content/drive/MyDrive/PKU_data/bone_fusion_per_class.csv
  scores_bone/ na Drive (cache bone)
